In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# For demonstration, we create a synthetic time series of 48 timesteps
# (imagine this is one feature of one sample from your X with shape (48,))
np.random.seed(42)
time_steps = 48
t = np.linspace(0, 4*np.pi, time_steps)
# A synthetic signal: sine wave plus noise
signal = np.sin(t) + 0.1 * np.random.randn(time_steps)

# Define compression factor F; for 48 timesteps and F=4, we get 12 segments.
F = 4
n_segments = time_steps // F

def encoder(signal, F):
    """
    Simulate an encoder by splitting the signal into segments of length F
    and computing the average of each segment.
    """
    latent = np.array([np.mean(signal[i * F:(i + 1) * F]) for i in range(n_segments)])
    return latent

# Simulate encoding: get a latent representation for each segment.
latent = encoder(signal, F)
print("Latent representation:", latent)

# Assume a simple codebook for vector quantization. In practice, these codewords are learned.
# For simplicity, we define a small codebook with 4 possible tokens.
codebook = np.array([ -0.8, 0.0, 0.8, 1.6 ])  # Example codewords

def quantize(latent, codebook):
    """
    For each latent value, find the index of the nearest codeword.
    """
    tokens = []
    for val in latent:
        distances = np.abs(codebook - val)
        token = np.argmin(distances)
        tokens.append(token)
    return np.array(tokens)

# Quantize the latent representation to get a sequence of tokens.
tokens = quantize(latent, codebook)
print("Tokens:", tokens)

# Now simulate the decoder.
def decoder(tokens, codebook, F):
    """
    Decode the token sequence back into a continuous time series by
    replacing each token with its corresponding codeword and repeating it F times.
    """
    decoded = []
    for token in tokens:
        decoded.extend([codebook[token]] * F)
    return np.array(decoded)

# Decode the tokens to reconstruct the signal.
decoded_signal = decoder(tokens, codebook, F)

# Now, let’s apply masking.
# For example, we decide to mask tokens in positions 4 to 6 (0-indexed).
mask_indices = np.arange(4, 7)
# Define a baseline token, e.g., we could use the codebook's middle value (or a dedicated mask token)
baseline_token = 1  # For instance, token index 1 corresponding to codebook[1]
masked_tokens = tokens.copy()
masked_tokens[mask_indices] = baseline_token

# Decode the masked tokens
decoded_masked_signal = decoder(masked_tokens, codebook, F)

# Plot the original signal, decoded (tokenized) signal, and masked-decoded signal
plt.figure(figsize=(10, 6))
plt.plot(signal, label="Original Signal", marker='o')
plt.plot(decoded_signal, label="Decoded from Tokens", marker='x')
plt.plot(decoded_masked_signal, label="Masked (Decoded) Signal", marker='s')
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.title("TOTEM-style Tokenization and Masking Example")
plt.legend()
plt.show()


Latent representation: [ 0.43638814  0.99802209  0.51956673 -0.53006728 -1.03080269 -0.49005773
  0.45690515  0.96428588  0.39436652 -0.61302156 -0.93873609 -0.413206  ]
Tokens: [2 2 2 0 0 0 2 2 1 0 0 0]
